Previamente crear azure sql y los secretscope en el ambiente dev o prod

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_ecobici_dev")
dbutils.widgets.text("sqlServer", "serverecobici.database.windows.net")
dbutils.widgets.text("sqlDatabase", "dbecobici")
dbutils.widgets.text("secretScope", "kv-ecobici")
dbutils.widgets.text("secretUsuario", "sql-usuario")
dbutils.widgets.text("secretPassword", "sql-password")
dbutils.widgets.text("esquemaSql", "dbo")
dbutils.widgets.dropdown("exportarFact", "no", ["si", "no"])

In [0]:
catalogo      = dbutils.widgets.get("catalogo")
sql_server    = dbutils.widgets.get("sqlServer").strip()
sql_database  = dbutils.widgets.get("sqlDatabase").strip()
esquema_sql   = dbutils.widgets.get("esquemaSql").strip()
exportar_fact = dbutils.widgets.get("exportarFact") == "si"

In [0]:
assert sql_server and sql_database, "Llena los widgets sqlServer y sqlDatabase"

In [0]:
USUARIO = dbutils.secrets.get(dbutils.widgets.get("secretScope"), dbutils.widgets.get("secretUsuario"))
CLAVE   = dbutils.secrets.get(dbutils.widgets.get("secretScope"), dbutils.widgets.get("secretPassword"))

In [0]:
URL = (
    f"jdbc:sqlserver://{sql_server}:1433;database={sql_database};"
    "encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=60;"
)
DRIVER = "com.microsoft.sqlserver.jdbc.SQLServerDriver"

In [0]:
TABLAS_PEQUENAS = [
    "dim_fecha", "dim_estacion", "dim_usuario", "dim_clima", "fact_clima_hora",
    "agg_demanda_hora_dia", "agg_estaciones", "agg_rutas_frecuentes", "agg_uso_por_clima", "agg_resumen_diario",
]
TABLAS_GRANDES = {"fact_viajes": 4}      # tabla -> número de conexiones en paralelo

TIPOS_ESPECIALES = {"viaje_id": "CHAR(64)", "nombre_estacion": "VARCHAR(400)", "genero": "CHAR(2)"}

In [0]:
def tipos_sql(df):
    columnas = [
        f"{c.name} {TIPOS_ESPECIALES.get(c.name, 'VARCHAR(200)')}"
        for c in df.schema.fields
        if c.dataType.simpleString() == "string"
    ]
    return ", ".join(columnas)

In [0]:
def leer_sql(consulta):
    return (
        spark.read.format("jdbc")
        .options(url=URL, dbtable=f"({consulta}) AS t", user=USUARIO, password=CLAVE, driver=DRIVER)
        .load()
    )

1. Probar conexión

In [0]:
display(leer_sql("SELECT @@VERSION AS version_sql, DB_NAME() AS base_de_datos"))

2. Exportar

In [0]:
def exportar(nombre, particiones=1):
    df = spark.table(f"{catalogo}.golden.{nombre}")
    opciones = {
        "url": URL,
        "dbtable": f"{esquema_sql}.{nombre}",
        "user": USUARIO,
        "password": CLAVE,
        "driver": DRIVER,
        "truncate": "true",
        "batchsize": "10000",
        "numPartitions": str(particiones),
    }
    tipos = tipos_sql(df)
    if tipos:
        opciones["createTableColumnTypes"] = tipos

    df.repartition(particiones).write.format("jdbc").options(**opciones).mode("overwrite").save()
    print(f"Exportada {esquema_sql}.{nombre}: {df.count():,} filas")


for nombre in TABLAS_PEQUENAS:
    exportar(nombre)

In [0]:
if exportar_fact:
    for nombre, particiones in TABLAS_GRANDES.items():
        exportar(nombre, particiones)
else:
    print("exportarFact = no: se omite fact_viajes. Cámbialo a 'si' y vuelve a ejecutar esta celda.")

3. Verificar que SQL tenga las mismas filas

In [0]:
pendientes = TABLAS_PEQUENAS + (list(TABLAS_GRANDES) if exportar_fact else [])
for nombre in pendientes:
    en_spark = spark.table(f"{catalogo}.golden.{nombre}").count()
    en_sql = leer_sql(f"SELECT COUNT_BIG(*) AS n FROM {esquema_sql}.{nombre}").first()["n"]
    estado = "OK" if en_spark == en_sql else "DIFERENTE"
    print(f"{nombre:24s} Databricks: {en_spark:>12,} | Azure SQL: {en_sql:>12,} | {estado}")
    assert en_spark == en_sql, f"{nombre}: las filas no coinciden"